# The contrast ladder: every comparison a study supports, each against its own baseline


## What went wrong first

Control pairing used to ask one question of each sample group: *does this label look like a control?* So it read `WT` in `T. gondii VEG WT, Switch Conditions` and marked the **alkaline-stressed** arm of GSE218921 as a baseline.

`WT` is a level on the **strain** axis. The baseline for a stress question lives on the **treatment** axis. Having claimed the treated group as the control, nothing was left to pair it with, so the contrast was never formed and no column for it entered the information map -- absent rather than wrong.

The cost was the cleanest bradyzoite-switch signal in the collection: BFD1 +1.36, BAG1 +4.75, ENO1 +4.91, SAG1 -1.34. The study fell back to internal normalisation against a centre built from all six groups mixed, switch samples included, so part of the stress response sat in the baseline and cancelled -- which is the mechanism behind cross-study agreement on alkaline stress of 0.236.

The fix is not a longer keyword list. It is to parse the design and let the baselines nest.

In [1]:
import importlib.util, csv, collections
spec = importlib.util.spec_from_file_location('cl', '/media/carruthers/mnt3/claude/repo/starplast/scripts/build_contrast_ladder.py')
CL = importlib.util.module_from_spec(spec); spec.loader.exec_module(CL)
rows = list(csv.DictReader(open('/media/carruthers/mnt3/claude/repo/starplast/results/rna_sample_quality_2026_11_04/sample_quality.tsv'), delimiter='\t'))
len(rows), len({r['study'] for r in rows})

(423, 31)

## Step 1: the design is already written down

GEO submitters record the structured part as `axis: level` pairs separated by `;`. The split is never on `,` -- a treatment level legitimately contains commas, as `high pH, low C02, low serum conditions` does.

In [2]:
groups = collections.defaultdict(set)
for r in rows: groups[r['study']].add(r['condition_group'])
axes = collections.Counter()
structured = 0
for gs in groups.values():
    for g in gs:
        f, _ = CL.parse_factors(g)
        structured += bool(f)
        axes.update(f.keys())
print('groups with a parseable design: %d of %d' % (structured, sum(len(v) for v in groups.values())))
axes.most_common()

groups with a parseable design: 178 of 209


[('treatment', 140), ('genotype', 104), ('strain', 90), ('tissue', 54), ('time', 16), ('stage', 14)]

## Step 2: the axis a depositor left in free text

GSE218921 is the case that matters. All six tachyzoite groups carry an **identical** structured head -- `strain: VEG; treatment: ...; tissue: ...`. The genotype distinguishing WT from ROCY1 KO and BFD1 KO exists only in the sample name. Without recovering it the three strains collapse into one group and no genotype axis is seen at all.

In [3]:
for g in sorted(groups['GSE218921']):
    f, tail = CL.parse_factors(g)
    if 'genotype' not in f:
        f['genotype'] = CL.genotype_from_tail(tail) or 'unstated'
    print('%-12s %-34s %s' % (f['genotype'], f['treatment'][:34], f['tissue'][:40]))

unstated     control media conditions           freshly excysted sporozoites
bfd1 ko      control media conditions           replicating tachyzoites under different 
rocy1 ko     control media conditions           replicating tachyzoites under different 
wild type    control media conditions           replicating tachyzoites under different 
unstated     high ph, low c02, low serum condit freshly excysted sporozoites
bfd1 ko      high ph, low c02, low serum condit replicating tachyzoites under different 
rocy1 ko     high ph, low c02, low serum condit replicating tachyzoites under different 
wild type    high ph, low c02, low serum condit replicating tachyzoites under different 


## Step 3: pair on exactly one axis, and orient by the reference level

Two groups are comparable when they differ on **one** axis. The group sitting at that axis reference level -- untreated, wild type, earliest time -- is the baseline, and every row records the word that decided it.

Strain, tissue and stage never get an automatic reference. Choosing a strain is choosing an organism, not choosing a control, and silently making ME49 the baseline is how a strain difference gets reported as a treatment effect.

In [4]:
contrasts, unresolved, refused = CL.build(rows, 'condition_group', 'study')
print('contrasts: %d across %d studies' % (len(contrasts), len({c['study'] for c in contrasts})))
collections.Counter(c['axis'] for c in contrasts).most_common()

contrasts: 85 across 15 studies


[('genotype', 37), ('treatment', 27), ('time', 21)]

## The ladder for GSE218921

`depth` counts how many **other** axes sit off their own reference. Depth 0 is the clean effect on an otherwise-baseline background; depth 1 is the same effect inside a mutant or a stressed background. Nothing is averaged across depths -- the treatment effect inside a knockout is a different measurement from the effect in wild type.

The first treatment row is the contrast the keyword rule deleted. The depth-1 treatment rows are the nesting the design calls for: each knockout measured against its own untreated cells. The depth-1 genotype rows are the interaction terms, which the construction gives for free.

In [5]:
for c in sorted((c for c in contrasts if c['study'] == 'GSE218921'),
                key=lambda c: (c['axis'], c['depth'])):
    print('d%s %-9s %-26s -> %-22s [%s]' % (c['depth'], c['axis'], c['baseline_level'][:26],
                                           c['treated_level'][:22], c['background'][:52]))

d0 genotype  wild type                  -> bfd1 ko                [tissue=replicating tachyzoites under different condi]
d0 genotype  wild type                  -> rocy1 ko               [tissue=replicating tachyzoites under different condi]
d1 genotype  wild type                  -> bfd1 ko                [treatment=high ph, low c02, low serum conditions; ti]
d1 genotype  wild type                  -> rocy1 ko               [treatment=high ph, low c02, low serum conditions; ti]
d0 treatment control media conditions   -> high ph, low c02, low  [genotype=wild type; tissue=replicating tachyzoites u]
d1 treatment control media conditions   -> high ph, low c02, low  [genotype=unstated; tissue=freshly excysted sporozoit]
d1 treatment control media conditions   -> high ph, low c02, low  [genotype=bfd1 ko; tissue=replicating tachyzoites und]
d1 treatment control media conditions   -> high ph, low c02, low  [genotype=rocy1 ko; tissue=replicating tachyzoites un]


## Refusing rather than guessing

A pair is formed only when the text no parsed factor explains is the same on both sides. If it differs, the study varies on an axis nobody parsed, and pairing across it compares two things at once. GSE132238 declares genotype and timepoint while `stress`/`unstress` hides in the sample name; without this rule `WT_stress_24h` is compared to `KO_unstress_24h` and a stress response is reported as a genotype effect.

Leftover words are attributed to an axis by asking the study rather than a word list: a word belongs to axis A when its presence is constant inside every level of A. That recognises `Switch Conditions` as a restatement of `treatment: high pH` while leaving `stress` unexplained and therefore disqualifying.

In [6]:
print('pairs refused: %d (real axis %d, plate or batch label %d)' % (
    len(refused),
    sum(1 for r in refused if r['looks_like'] == 'a real condition axis'),
    sum(1 for r in refused if r['looks_like'] == 'plate or batch identifier')))
collections.Counter(r['unparsed_difference'] for r in refused
                    if r['looks_like'] == 'a real condition axis').most_common(8)

pairs refused: 168 (real axis 36, plate or batch label 132)


[('stress / unstress', 18), ('riboseq / rnaseq', 8), ('etoh / shield1', 4), ('gra18 / pru', 2), ('bystander / infected', 2), ('us / ut', 1), ('pru / rh / us / ut', 1)]

## Two measurements must never print as one row

Where an unparsed axis is the only thing separating two contrasts, it is named in the background (`unparsed=stress`) rather than dropped. That keeps the rows distinct and tells the reviewer exactly what needs naming. The count below must stay zero: identical rows get averaged together downstream.

In [7]:
sig = collections.Counter((c['study'], c['axis'], c['baseline_level'], c['treated_level'],
                           c['background']) for c in contrasts)
sum(v for v in sig.values() if v > 1)

0

## What is left for a person

Two kinds: axes where no level matches the reference vocabulary (`etoh | shield1` -- Shield-1 stabilisation against an ethanol vehicle), and axes that exist only in free text (`bystander | infected`, `gra18 | pru`).

This is the review item 79 described, an order of magnitude smaller: naming the reference level of these axes rather than naming the control for 450 runs.

In [8]:
for u in unresolved[:12]:
    print('%-12s %-22s %s' % (u['study'], u['axis'][:22], u['levels'][:54]))
len(unresolved)

GSE103113    UNPARSED (free text)   gra18 | pru
GSE116539    strain                 tgcrt ko derived from rh{delta}ku80{delta}hxg clone | 
GSE125120    UNPARSED (free text)   bystander | infected
GSE132238    UNPARSED (free text)   stress | unstress
GSE136123    UNPARSED (free text)   pru | rh | us | ut
GSE136123    genotype               morc kd | unstated
GSE139537    UNPARSED (free text)   etoh | shield1
GSE159515    UNPARSED (free text)   a | b
GSE159515    strain                 me49 | rh
GSE159515    stage                  bradyzoites | tachyzoites
GSE178355    strain                 rh::tir1::mettl3-aid | rh::tir1::wtap-aid | rh::tir1::
GSE178355    genotype               mettl3- aid | wtap- aid | yth1- aid


30

## Gate

Half met. The gate in instruction 79 is that cross-study agreement on alkaline stress rises from 0.236 **and** the GSE218921 contrast exists as a column. The contrast is built and correctly oriented, which is the second half. The first half needs the expression columns rebuilt from this ladder, which is the next step and is not in this notebook.